# H3b: ERA5-Land against CERRA-Land over Belgium

Ticket #82, first data step (13/11/2026): the monthly ratio of ERA5-Land to CERRA-Land over Belgium, 1/2012–11/2025,
on the same 06–06 UTC days, to look for steps of either bridge (above all at 5–7/2021, the end of the original
CERRA-Land production, and at 7/2024, the ODYSSEY → CIRRUS change), and a check of the 12-bit packing of
5/2021–7/2025 on CERRA-Land precipitation. The CERRA-Land variant of the correction waits for G1 (H3, #20).
Sources: `docs/sources/` (2016 Soci, 2021 Le Moigne, 2024 Ridal, 2024 Schimanke, 2025 Wood, 2026 C3S, 2026 Marti).
Results in `docs/method.md`, *ERA5-Land against CERRA-Land (H3b)*.

In [ ]:
from opera_ds.paths import DATA_DIR

import calendar
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import shapely
import xarray as xr
from scipy.stats import kendalltau, linregress, mannwhitneyu, ttest_ind

sys.path.insert(0, str(Path.cwd().parent))          # scripts/ is not a package of opera_ds
from scripts.make_region_mask import country_geometry

FIG = Path.cwd().parent / "docs" / "figures" / "bridge"
geom = country_geometry("BEL", str(DATA_DIR / "naturalearth"))

## 1. Daily amounts on the Belgian cells

Cells whose centre is inside Belgium (Natural Earth 1:10m), each bridge on its own grid. ERA5-Land hourly amounts
(the hour ending at `valid_time`) are summed into the CERRA-Land days: the day labelled *d* 06 UTC takes the hours
ending in (*d*−1 06 UTC, *d* 06 UTC]. One coastal ERA5-Land cell has no values (sea) and is left out; the first
day (1/1/2012 06 UTC) has 7 hours only and is left out.

In [ ]:
e = xr.open_zarr(DATA_DIR / "era5land_bel.zarr", consolidated=False)
c = xr.open_zarr(DATA_DIR / "cerraland_bel.zarr", consolidated=False)
LO, LA = np.meshgrid(e.longitude.values, e.latitude.values)
ebel = shapely.contains_xy(geom, LO, LA)
cbel = shapely.contains_xy(geom, c.longitude.values, c.latitude.values)
yi, xi = np.where(ebel)
box = dict(latitude=slice(yi.min(), yi.max() + 1), longitude=slice(xi.min(), xi.max() + 1))
m = ebel[box["latitude"], box["longitude"]]
days = c.valid_time.values.astype("datetime64[h]")
win = e.isel(**box).sel(valid_time=slice(days[0] - np.timedelta64(23, "h"), days[-1]))
tp = win["tp"].values[:, m].astype(np.float64)
t = win["valid_time"].values.astype("datetime64[h]")
assert np.all(win["expver"].values == 1), "preliminary ERA5-Land hours in the window"
lab = (t - np.timedelta64(7, "h")).astype("datetime64[D]").astype("datetime64[h]") + np.timedelta64(30, "h")
assert np.array_equal(np.unique(lab), days)
inv = np.searchsorted(days, lab)
era_day = np.zeros((days.size, tp.shape[1])); np.add.at(era_day, inv, tp)
hours = np.bincount(inv, minlength=days.size)
keep = ~np.isnan(era_day).all(0)
era_day = era_day[:, keep]
cer_day = c["tp"].values[:, cbel].astype(np.float64)
ok_day = (hours == 24) & ~np.isnan(era_day).any(1)
print(f"Belgian cells: ERA5-Land {keep.sum()} (of {ebel.sum()}), CERRA-Land {cbel.sum()}; days {days.size}, "
      f"complete {ok_day.sum()}; CERRA-Land missing values {np.isnan(cer_day).sum()}")

## 2. Monthly ratio

A month is the window (1st 06 UTC, 1st of the next month 06 UTC], as `opera_ds.bridge.cerraland_sum`; months with a
missing day are left out (12/2025, whose last day is labelled 1/1/2026). The ratio is of the area means of each
grid; the weather cancels in it, so a step points to a change in one of the two (*Homogeneity of the bridge*, H3).

In [ ]:
E, C = era_day.mean(1), cer_day.mean(1)
mon = (days - np.timedelta64(30, "h")).astype("datetime64[M]")
keys, e_m, c_m = [], [], []
for mm in np.unique(mon):
    k = mon == mm
    y, mo = int(str(mm)[:4]), int(str(mm)[5:7])
    if k.sum() == calendar.monthrange(y, mo)[1] and ok_day[k].all():
        keys.append(str(mm)); e_m.append(E[k].sum()); c_m.append(C[k].sum())
e_m, c_m = np.array(e_m), np.array(c_m)
r = np.log10(e_m / c_m)
yrs = np.array([int(k[:4]) for k in keys]); mos = np.array([int(k[5:]) for k in keys])
print(f"{len(keys)} months, {keys[0]} → {keys[-1]}; ERA5-Land / CERRA-Land over all: ×{e_m.sum() / c_m.sum():.3f}; "
      f"monthly spread (sd of log10) {r.std(ddof=1):.3f}")

## 3. Steps at the dates known from the metadata

Same test as H3 (`opera_ds.homog.step` on the H3 branch, #79, repeated here until H3 is merged): mean log10 ratio of
the n months from the date against the n months before, two-sample t-test and Wilcoxon–Mann–Whitney (WMO 2020;
Ribeiro et al. 2016). Consecutive months may be autocorrelated, which makes the p-values too small. Dates: 5/2021
(first month of the 12-bit packing and, by the data user guide, the end of the original period), 7/2021 (after
the end of the original production in 6/2021, Ridal et al. 2024), 7/2024 (ODYSSEY → CIRRUS).

In [ ]:
def step(at, n):
    i = keys.index(at)
    b, a = r[max(0, i - n):i], r[i:i + n]
    return dict(step=a.mean() - b.mean(), sd=b.std(ddof=1), nb=len(b), na=len(a),
                t_p=ttest_ind(a, b).pvalue, mw_p=mannwhitneyu(a, b).pvalue)

print("date     n  | step log10 (factor) | sd before | months | t-test p | MW p")
for at in ("2021-05", "2021-07", "2024-07"):
    for n in (12, 18, 24):
        s = step(at, n)
        print(f"{at}  {n:2d} | {s['step']:+.3f} (×{10 ** s['step']:.2f})    | {s['sd']:.3f}     | {s['nb']}/{s['na']}  "
              f"| {s['t_p']:.2f}     | {s['mw_p']:.2f}")

## 4. A drift rather than a step

The yearly ratios, by season, and the linear trend of the monthly log ratio; then, as an exploration only, the
12-month step at every month (many tests: the smallest p-values are expected to be small by chance).

In [ ]:
print("year  all   DJF   MAM   JJA   SON   ERA5-Land  CERRA-Land (mm)")
for y in range(2012, 2026):
    row = [e_m[yrs == y].sum() / c_m[yrs == y].sum()]
    for ss in ([12, 1, 2], [3, 4, 5], [6, 7, 8], [9, 10, 11]):
        k = (yrs == y) & np.isin(mos, ss)
        row.append(e_m[k].sum() / c_m[k].sum())
    full = (yrs == y).sum() == 12
    print(f"{y}  " + "  ".join(f"{v:.2f}" for v in row)
          + (f"   {e_m[yrs == y].sum():6.0f}  {c_m[yrs == y].sum():6.0f}" if full else "   (11 months)"))
lr = linregress(np.arange(r.size) / 12, r)
ann = np.array([np.log10(e_m[yrs == y].sum() / c_m[yrs == y].sum()) for y in range(2012, 2025)])
tau = kendalltau(np.arange(ann.size), ann)
print(f"trend of the monthly log ratio: {lr.slope:+.4f} log10 a year (×{10 ** (10 * lr.slope):.2f} a decade), "
      f"p {lr.pvalue:.1e}; Kendall tau of the yearly ratios 2012–2024 {tau.statistic:.2f}, p {tau.pvalue:.4f}")
scan = sorted(((ttest_ind(r[i:i + 12], r[i - 12:i]).pvalue, keys[i], r[i:i + 12].mean() - r[i - 12:i].mean())
               for i in range(12, r.size - 12)))
print("12-month steps with the smallest p:", [(k, f"{s:+.3f}", f"{p:.3f}") for p, k, s in scan[:5]])

In [ ]:
ti = np.array([np.datetime64(k) for k in keys])
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ti, 10 ** r, color="0.6", lw=0.8, label="month")
run = np.convolve(r, np.ones(12) / 12, mode="valid")
ax.plot(ti[11:], 10 ** run, color="C0", lw=2, label="12 months ending at the month")
for d, txt in (("2021-05", "5/2021"), ("2024-07", "7/2024")):
    ax.axvline(np.datetime64(d), color="C3", ls="--", lw=1)
    ax.text(np.datetime64(d), 2.45, " " + txt, color="C3", va="top")
ax.axhline(1, color="k", lw=0.5)
ax.set_yscale("log"); ax.set_ylim(0.6, 2.5)
ax.set_yticks([0.7, 1, 1.4, 2]); ax.set_yticklabels(["0.7", "1", "1.4", "2"]); ax.yaxis.set_minor_formatter(plt.NullFormatter())
ax.set_ylabel("ERA5-Land / CERRA-Land")
ax.set_title("Belgium, monthly totals on 06–06 UTC days, 1/2012–11/2025")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FIG / "era5land-cerraland-ratio.png", dpi=120)

## 5. The 12-bit packing of 5/2021–7/2025

The data user guide names snow as the most affected variable and says nothing on precipitation. A 12-bit packing
of a daily field would leave about 4,096 possible values a day, spaced by about a 4,096th of the day's range. Here:
the smallest gap between distinct values, the number of distinct values a day (one day a week) and the share of
small amounts, on the whole window, before, during and after.

In [ ]:
tpc, tc = c["tp"].values, c.valid_time.values
periods = {"1/2012–4/2021": tc < np.datetime64("2021-05-01"),
           "5/2021–7/2025": (tc >= np.datetime64("2021-05-01")) & (tc < np.datetime64("2025-08-01")),
           "8/2025–12/2025": tc >= np.datetime64("2025-08-01")}
for name, k in periods.items():
    sub = tpc[k][::7]
    gap = np.nanmedian([np.diff(np.unique(x[x > 0])).min() for x in sub])
    nval = np.median([np.unique(x).size for x in sub])
    print(f"{name:15s} | smallest gap {gap:.1e} mm | distinct values a day {nval:6.0f} | "
          f"0 < tp < 0.1 mm {np.mean((sub > 0) & (sub < 0.1)):.3f} | dry {np.mean(sub == 0):.3f}")

## Findings

See `docs/method.md`, *ERA5-Land against CERRA-Land (H3b)*.